# Validation – Web Scraping vs. API

The scraped Curia Vista data is compared with the Open Data Web Services of the Swiss Parliament (http://ws-old.parlament.ch).

- **Input:** raw scraped data `01_Collection/data/curia_vista_anfragen_erledigt.csv` (test sample, 20 business items)
- **API endpoints:** `/affairs/{AffairId}` (business item) and `/councillors/{CouncillorId}` (party)
- **Output:** `04_Validation/data/validation_results.csv`

In [1]:
import time
from pathlib import Path

import pandas as pd
import requests

API_URL = "http://ws-old.parlament.ch"
HEADERS = {"Accept": "application/json"}
REQUEST_DELAY = 0.5  # seconds between requests

INPUT_FILE = Path("../01_Collection/data/curia_vista_anfragen_erledigt.csv")
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

scraped = pd.read_csv(INPUT_FILE, dtype={"business_number": str})
scraped["cosigners"] = scraped["cosigners"].fillna("")
print(f"Scraped business items: {len(scraped)}")

Scraped business items: 20


## 1. Retrieve the business items from the API

In [2]:
def get_json(endpoint):
    response = requests.get(f"{API_URL}/{endpoint}", params={"format": "json", "lang": "de"},
                            headers=HEADERS, timeout=30)
    response.raise_for_status()
    time.sleep(REQUEST_DELAY)
    return response.json()


def parse_affair(affair):
    """Extract the fields that are also available on the website."""
    author = affair.get("author") or {}
    councillor = author.get("councillor") or {}
    cosigners = [role["councillor"]["name"] for role in affair.get("roles", [])
                 if role.get("type") == "cosign" and role.get("councillor")]
    departments = [d["abbreviation"] for draft in affair.get("drafts", [])
                   for d in draft.get("relatedDepartments", [])]
    return {
        "api_business_number": affair.get("shortId"),
        "api_business_type": (affair.get("affairType") or {}).get("name"),
        "api_submitted_by": councillor.get("name"),
        "api_councillor_id": councillor.get("id"),
        "api_parliamentary_group": (author.get("faction") or {}).get("name"),
        "api_submission_date": pd.to_datetime(affair["deposit"]["date"]).date(),
        "api_submitted_in": affair["deposit"]["council"]["name"],
        "api_state": (affair.get("state") or {}).get("name"),
        "api_departments": departments,
        "api_cosigners": cosigners,
        "api_topic_codes": affair.get("additionalIndexing"),
    }


api_rows = []
for affair_id in scraped["affair_id"]:
    try:
        api_rows.append({"affair_id": affair_id, **parse_affair(get_json(f"affairs/{affair_id}"))})
    except requests.RequestException as e:
        print(f"{affair_id}: request failed ({e})")
        api_rows.append({"affair_id": affair_id})

api = pd.DataFrame(api_rows)
print(f"Retrieved from API: {api['api_business_number'].notna().sum()} of {len(api)}")

Retrieved from API: 20 of 20


## 2. Party of the submitter
The business item in the API only contains the parliamentary group. The party is taken from `/councillors/{CouncillorId}`.
**Note:** this endpoint may return the *current* party of the councillor, not the party at the time of submission.

In [3]:
party_by_id = {}
for councillor_id in api["api_councillor_id"].dropna().astype(int).unique():
    try:
        party_by_id[councillor_id] = get_json(f"councillors/{councillor_id}").get("partyName")
    except requests.RequestException as e:
        print(f"{councillor_id}: request failed ({e})")

api["api_party"] = api["api_councillor_id"].map(party_by_id)

## 3. Compare website and API field by field

In [4]:
df = scraped.merge(api, on="affair_id", how="left")
df["submission_date"] = pd.to_datetime(df["submission_date"], format="%d.%m.%Y").dt.date


def same_cosigners(row):
    website = set(filter(None, row["cosigners"].split("; ")))
    return website == set(row["api_cosigners"] or [])


checks = {
    "business_number": df["business_number"] == df["api_business_number"],
    "business_type": df["business_type"] == df["api_business_type"],
    "submitted_by": df["submitted_by"] == df["api_submitted_by"],
    "councillor_id": df["councillor_id"] == df["api_councillor_id"],
    "parliamentary_group": df["parliamentary_group"] == df["api_parliamentary_group"],
    "party": df["party"] == df["api_party"],
    "submission_date": df["submission_date"] == df["api_submission_date"],
    "submitted_in": df["submitted_in"] == df["api_submitted_in"],
    "state": df["state"] == df["api_state"],
    "responsible_authority": df.apply(
        lambda r: any(f"({abbr})" in str(r["responsible_authority"]) for abbr in r["api_departments"] or []), axis=1),
    "cosigners": df.apply(same_cosigners, axis=1),
}

for field, match in checks.items():
    df[f"match_{field}"] = match

summary = pd.DataFrame({
    "matches": {field: int(match.sum()) for field, match in checks.items()},
    "mismatches": {field: int((~match).sum()) for field, match in checks.items()},
})
summary["match_%"] = (summary["matches"] / len(df) * 100).round(1)
summary

,matches,mismatches,match_%
business_number,20,0,100.0
business_type,20,0,100.0
submitted_by,20,0,100.0
councillor_id,20,0,100.0
parliamentary_group,16,4,80.0
party,20,0,100.0
submission_date,20,0,100.0
submitted_in,20,0,100.0
state,20,0,100.0
responsible_authority,20,0,100.0


## 4. Mismatches in detail

In [5]:
rows = []
for field in checks:
    api_col = f"api_{field}" if field != "responsible_authority" else "api_departments"
    if field == "cosigners":
        api_col = "api_cosigners"
    for _, row in df[~df[f"match_{field}"]].iterrows():
        rows.append({"business_number": row["business_number"], "field": field,
                     "website": row[field], "api": row[api_col]})

mismatches = pd.DataFrame(rows, columns=["business_number", "field", "website", "api"])
print(f"Mismatches: {len(mismatches)}")
mismatches

Mismatches: 4


,business_number,field,website,api
0,26.1031,parliamentary_group,FDP-Liberale Fraktion,NaN
1,26.1030,parliamentary_group,FDP-Liberale Fraktion,NaN
2,26.1028,parliamentary_group,Sozialdemokratische Fraktion,NaN
3,26.1020,parliamentary_group,Sozialdemokratische Fraktion,NaN


## 5. Topics
The API does not return topic names, only codes in `additionalIndexing`. A direct comparison is therefore not possible yet. As a first indication, the number of codes is compared with the number of scraped topics (not verified that the codes correspond to the topics).

In [6]:
df["topic_count_website"] = df["topics"].fillna("").apply(lambda s: len([t for t in s.split(";") if t.strip()]))
df["topic_count_api"] = df["api_topic_codes"].fillna("").apply(lambda s: len([c for c in s.split(";") if c.strip()]))
df["match_topic_count"] = df["topic_count_website"] == df["topic_count_api"]

print(f"Same number of topics/codes: {df['match_topic_count'].sum()} of {len(df)}")
df[["business_number", "topics", "api_topic_codes", "topic_count_website", "topic_count_api"]]

Same number of topics/codes: 20 of 20


,business_number,topics,api_topic_codes,topic_count_website,topic_count_api
0,26.1035,Internationale Politik; Kultur,2831;08,2,2
1,26.1034,Internationale Politik; Sozialer Schutz; Staat...,04;08;2836,3,3
2,26.1032,Raumplanung und Wohnungswesen; Staatspolitik,2846;04,2,2
3,26.1031,Finanzwesen; Medien und Kommunikation; Staatsp...,15;34;24;04,4,4
4,26.1030,Beschäftigung und Arbeit; Steuer,44;2446,2,2
5,26.1029,Menschenrechte; Sozialer Schutz; Strafrecht,1236;2836;1216,3,3
6,26.1028,Soziale Fragen,28,1,1
7,26.1027,Soziale Fragen,28,1,1
8,26.1026,Beschäftigung und Arbeit; Soziale Fragen,28;44,2,2
9,26.1025,Gesundheit; Soziale Fragen,2841;28,2,2


## Save

In [7]:
df.to_csv(OUTPUT_DIR / "validation_results.csv", index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows to {OUTPUT_DIR / 'validation_results.csv'}")

Saved 20 rows to data/validation_results.csv


## Summary
*To be written by the group after reviewing the results above (which fields match, which differences exist and how they are handled).*